# Ouroboros Battle Test Report

| Field | Value |
|-------|-------|
| Session ID | `bt-2026-04-27-205831` |
| Stop Reason | `idle_timeout` |
| Duration | 1449.4 s |


In [ ]:
import json
import math

# Summary data embedded directly — notebook is self-contained
_SUMMARY_JSON = '''
{
  "schema_version": 2,
  "session_id": "bt-2026-04-27-205831",
  "stop_reason": "idle_timeout",
  "duration_s": 1449.381227016449,
  "stats": {
    "attempted": 2,
    "completed": 0,
    "failed": 2,
    "cancelled": 0,
    "queued": 0
  },
  "cost_total": 0.0,
  "cost_breakdown": {},
  "branch_stats": {
    "commits": 0,
    "files_changed": 0,
    "insertions": 0,
    "deletions": 0
  },
  "convergence_state": "INSUFFICIENT_DATA",
  "convergence_slope": 0.0,
  "convergence_r2": 0.0,
  "top_sensors": [
    [
      "governed_loop",
      2
    ]
  ],
  "top_techniques": [
    [
      "",
      2
    ]
  ],
  "operations": [
    {
      "op_id": "op-019dd0c2-f42d-7ca9-a862-586882db6099-cau",
      "status": "failed",
      "sensor": "governed_loop",
      "technique": "",
      "composite_score": 0.0,
      "elapsed_s": 91.76547841600001,
      "recorded_at": 1777324059.399533,
      "provider": "",
      "cost_usd": 0.0,
      "input_tokens": 0,
      "output_tokens": 0,
      "cached_tokens": 0,
      "tool_calls": 0,
      "files_changed": 1
    },
    {
      "op_id": "op-019dd0c2-f466-77e6-bbbb-782b2f544f75-cau",
      "status": "failed",
      "sensor": "governed_loop",
      "technique": "",
      "composite_score": 0.0,
      "elapsed_s": 137.689300042,
      "recorded_at": 1777324289.07631,
      "provider": "",
      "cost_usd": 0.0,
      "input_tokens": 0,
      "output_tokens": 0,
      "cached_tokens": 0,
      "tool_calls": 0,
      "files_changed": 1
    }
  ],
  "strategic_drift": {
    "total_ops": 25,
    "drifted_ops": 1,
    "ratio": 0.04,
    "threshold_met": true,
    "warning": false,
    "status": "ok"
  },
  "session_outcome": "complete",
  "last_activity_ts": 1777324960.65679,
  "cost_by_op_phase": {},
  "cost_by_op_phase_provider": {},
  "metrics": {
    "schema_version": 1,
    "session_id": "bt-2026-04-27-205831",
    "computed_at_unix": 1777324960.659481,
    "composite_score_session_mean": 0.0,
    "composite_score_session_min": 0.0,
    "composite_score_session_max": 0.0,
    "per_op_composite_scores": [
      0.0,
      0.0
    ],
    "trend": "INSUFFICIENT_DATA",
    "convergence_slope": 0.0,
    "convergence_oscillation_ratio": 0.0,
    "convergence_scores_analyzed": 2,
    "convergence_recommendation": "Fewer than 5 data points are available; no reliable trend can be inferred. Collect more iterations before acting on convergence signals.",
    "session_completion_rate": null,
    "self_formation_ratio": 0.0,
    "postmortem_recall_rate": 0.0,
    "cost_per_successful_apply": null,
    "posture_stability_seconds": null,
    "ops_inspected": 2,
    "ops_truncated": false,
    "notes": []
  },
  "metrics_observability_schema": "1.0"
}
'''

data = json.loads(_SUMMARY_JSON)

# Extract composite scores from operation_log
scores = [
    op['composite_score']
    for op in data.get('operation_log', [])
    if op.get('composite_score') is not None
]
print(f"Session: {data['session_id']}")
print(f"Scores extracted: {len(scores)}")
print(f"Scores: {scores}")


## Composite Score Trend

Plot of composite scores over operation index with a logarithmic fit overlay.

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

if len(scores) >= 2:
    x = np.arange(1, len(scores) + 1)
    y = np.array(scores)

    fig, ax = plt.subplots(figsize=(10, 5))
    ax.scatter(x, y, label='Composite Score', color='steelblue', zorder=3)
    ax.plot(x, y, color='steelblue', alpha=0.4)

    # Logarithmic fit overlay
    try:
        log_x = np.log(x)
        coeffs = np.polyfit(log_x, y, 1)
        fit_y = coeffs[0] * log_x + coeffs[1]
        ax.plot(x, fit_y, 'r--', label='Log fit', linewidth=2)
    except Exception as _e:
        print(f'Log fit failed: {_e}')

    ax.set_xlabel('Operation Index')
    ax.set_ylabel('Composite Score')
    ax.set_title('Composite Score Trend')
    ax.legend()
    ax.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.show()
else:
    print('Not enough scored operations to plot trend.')


## Convergence State

Analysis of score convergence based on logarithmic regression.

In [ ]:
convergence = data.get('convergence', {})
state = convergence.get('state', 'unknown')
slope = convergence.get('slope', 0.0)
r2 = convergence.get('r_squared_log', 0.0)

print(f'Convergence State : {state}')
print(f'Slope             : {slope:.6f}')
print(f'R² (log fit)      : {r2:.4f}')
print()

# Human-readable interpretation
if state == 'improving':
    print('Interpretation: The session shows a consistent improvement trend.')
elif state == 'converged':
    print('Interpretation: Scores have stabilised — further iterations are unlikely to help.')
elif state == 'stagnant':
    print('Interpretation: No meaningful progress detected; consider changing strategy.')
elif state == 'diverging':
    print('Interpretation: WARNING — scores are getting worse over time.')
else:
    print(f'Interpretation: Convergence state "{state}" is not recognised.')


## Operations Breakdown

Pie chart of operation outcomes.

In [ ]:
ops = data.get('operations', {})
labels = ['Completed', 'Failed', 'Cancelled', 'Queued']
values = [
    ops.get('completed', 0),
    ops.get('failed', 0),
    ops.get('cancelled', 0),
    ops.get('queued', 0),
]
colors = ['#4caf50', '#f44336', '#ff9800', '#2196f3']

# Filter out zero-value slices
pairs = [(l, v, c) for l, v, c in zip(labels, values, colors) if v > 0]
if pairs:
    _labels, _values, _colors = zip(*pairs)
    fig, ax = plt.subplots(figsize=(6, 6))
    ax.pie(_values, labels=_labels, colors=_colors, autopct='%1.1f%%', startangle=140)
    ax.set_title('Operations Breakdown')
    plt.tight_layout()
    plt.show()
else:
    print('No operation data available.')


## Sensor Activation

Horizontal bar chart of top sensor trigger counts.

In [ ]:
top_sensors = data.get('top_sensors', [])

if top_sensors:
    sensor_names = [s[0] for s in top_sensors]
    sensor_counts = [s[1] for s in top_sensors]

    fig, ax = plt.subplots(figsize=(8, max(3, len(sensor_names) * 0.6)))
    bars = ax.barh(sensor_names, sensor_counts, color='#7e57c2')
    ax.set_xlabel('Trigger Count')
    ax.set_title('Top Sensor Activations')
    ax.bar_label(bars, padding=3)
    ax.invert_yaxis()
    plt.tight_layout()
    plt.show()
else:
    print('No sensor data available.')


## Cost & Branch Summary

Breakdown of API costs and git branch statistics.

In [ ]:
cost = data.get('cost', {})
branch = data.get('branch', {})

print('=== Cost Summary ===')
print(f"Total cost : ${cost.get('total', 0.0):.4f}")
print('Breakdown  :')
for provider, amount in cost.get('breakdown', {}).items():
    print(f'  {provider:<30} ${amount:.4f}')

print()
print('=== Branch Summary ===')
print(f"Commits       : {branch.get('commits', 0)}")
print(f"Files changed : {branch.get('files_changed', 0)}")
print(f"Insertions    : {branch.get('insertions', 0)}")
print(f"Deletions     : {branch.get('deletions', 0)}")
